# Vocabularies — authority indexes + alignment cascade

Alignment of the fields admitted by `data/vocabularies/field_vocab_map.json` (N ≥ 100k occurrences, M ≥ 10 institutions), deduplicated on distinct `(group, data_source, value)`. Authority indexes are built once and cached; null markers and prose are routed out before matching; values are atomised on baseline plus induced separators; a house list is read first where an institution is credited with one; then exact (with homograph disambiguation and cheap retries), fuzzy, a zero-shot LLM atomiser, and retrieval plus LLM rerank. Places run the exact tier only. `object_name` appears nowhere here.

In [1]:
import gc
from functools import partial
from pathlib import Path
import codecs
import json
import re
import unicodedata
import xml.etree.ElementTree as ET

import numpy as np
import polars as pl
from codecarbon import EmissionsTracker
from rapidfuzz import fuzz
from rapidfuzz.process import cdist

from mds_norm.pipeline.build_local_termlists import load_termlist_index
from mds_norm.pipeline.build_standard_vocabs import BUILDERS as STANDARD_BUILDERS
from mds_norm.utils.inference import Inference
from mds_norm.utils.atomise import (NULL_MARKERS, NO_ATOMISE_FIELDS,
    PLACEHOLDER_MARKERS, SEMANTIC_MARKERS, morph_variants, PREFILTER, PROSE_MAX_CHARS,
    PROSE_MAX_TOKENS, ATOM_PROSE_MAX_CHARS, ATOM_PROSE_MAX_TOKENS,
    BASE_SEPARATORS, CANDIDATE_SEPARATORS, SEP_LITERAL, LLM_COVERAGE,
    COMPOUND_FIELDS, compound_head,
    LLM_MAX_NEW_TOKENS, DATE_LIKE, GROUP_DESC, LLM_PROMPT,
    parse_llm_atoms, atomise)

In [2]:
INTERMEDIATE_PATH = Path(".").resolve() / "analysis_output"
FIELD_STATS = INTERMEDIATE_PATH / "field_stats.parquet"
VOCAB_PATH = Path("../data/vocabularies").resolve()

OUT_DIR = INTERMEDIATE_PATH / "vocabularies"
OUT_DIR.mkdir(parents=True, exist_ok=True)
INDEX_DIR = OUT_DIR / "indexes"
INDEX_DIR.mkdir(parents=True, exist_ok=True)

EMISSIONS_LOG_PATH = INTERMEDIATE_PATH / "emissions_logs"
EMISSIONS_LOG_PATH.mkdir(parents=True, exist_ok=True)

FIELD_VOCAB_MAP = {
    "spectrum/" + field: vocabs
    for field, vocabs in json.loads((VOCAB_PATH / "field_vocab_map.json").read_text()).items()
}

# Agent fields link in 5_entities; here only their authority index
AGENT_FIELDS = [f for f, v in FIELD_VOCAB_MAP.items() if "ulan" in v]
GROUP_FOR = {f: "+".join(v) for f, v in FIELD_VOCAB_MAP.items() if f not in AGENT_FIELDS}
GROUP_VOCABS = {g: g.split("+") for g in set(GROUP_FOR.values())}
ALL_VOCABS = {v for vocabs in FIELD_VOCAB_MAP.values() for v in vocabs}
LOCAL_VOCABS = sorted(v for v in ALL_VOCABS if v.startswith("local_"))
STANDARD_VOCABS = sorted(v for v in ALL_VOCABS if v in STANDARD_BUILDERS)
PLACE_GROUPS = {g for g, v in GROUP_VOCABS.items() if "tgn" in v}
CONCEPT_GROUPS = set(GROUP_VOCABS) - PLACE_GROUPS

COMPONENT = "vocab_alignment"
# Base confidence per sub-component, scaled by the homograph settlement
CONFIDENCE = {"exact": 0.97, "exact_inst": 0.93, "exact_variant": 0.93,
              "exact_morph": 0.88, "exact_paren": 0.85, "exact_compound": 0.85,
              "fuzzy": 0.85,
              "rerank": 0.7, "llm_exact": 0.75, "llm_variant": 0.72,
              "llm_morph": 0.7}
RESOLVED_BY_FACTOR = {"unique": 1.0, "kind_tier": 0.95, "spatial": 0.92,
                      "prominent": 0.92, "margin": 1.0}

## 1. Authority indexes

One schema per vocabulary: `(subject, term, lang, kind, norm)`, `kind ∈ {prefLabelGVP, prefLabel, altLabel}`. Getty `2Terms` exports are parsed straight from N-Triples. Two side-tables support disambiguation: TGN direct-narrower counts (prominence) and PeriodO spatial coverage (UK first, then Europe). Indexes are cached under `analysis_output/vocabularies/indexes/` and returned as LazyFrames; delete a cache file to rebuild.

In [3]:
XL = "http://www.w3.org/2008/05/skos-xl#"
GVP = "http://vocab.getty.edu/ontology#"
NT_LINE = r"^<([^>]*)> <([^>]*)> (.*) \.$"
KIND_PRIORITY = {"prefLabelGVP": 0, "prefLabel": 1, "altLabel": 2}


def scan_nt(path: Path) -> pl.LazyFrame:
    # One triple per line, split on an absent byte
    return (
        pl.scan_csv(path, separator="\x00", has_header=False, new_columns=["line"],
                    quote_char=None)
        .select(
            s=pl.col("line").str.extract(NT_LINE, 1),
            p=pl.col("line").str.extract(NT_LINE, 2),
            o=pl.col("line").str.extract(NT_LINE, 3),
        )
    )


def unescape(col: pl.Expr) -> pl.Expr:
    return (
        pl.when(col.str.contains(r"\\"))
        .then(col.map_elements(
            lambda s: unicodedata.normalize("NFC", codecs.decode(s, "unicode_escape")),
            return_dtype=pl.String))
        .otherwise(col)
    )


def norm_term(col: pl.Expr) -> pl.Expr:
    return col.str.to_lowercase().str.replace_all(r"\s+", " ").str.strip_chars(" .,;:")


def getty_term_index(nt_path: Path) -> pl.LazyFrame:
    lf = scan_nt(nt_path)
    links = (
        lf.filter(pl.col("p").is_in([XL + "prefLabel", XL + "altLabel", GVP + "prefLabelGVP"]))
        .select(
            subject=pl.col("s").str.extract(r"/(\d+)$"),
            term_id=pl.col("o").str.extract(r"term/(\d+)"),
            kind=pl.col("p").str.extract(r"[#/](\w+)$"),
        )
    )
    terms = (
        lf.filter(pl.col("p") == GVP + "term")
        .select(
            term_id=pl.col("s").str.extract(r"term/(\d+)"),
            term=unescape(pl.col("o").str.extract(r'^"(.*)"(?:@[\w-]+)?$', 1)),
            lang=pl.col("o").str.extract(r"@([\w-]+)$"),
        )
    )
    return (
        links.join(terms, on="term_id")
        .select("subject", "term", "lang", "kind", norm=norm_term(pl.col("term")))
        .filter(pl.col("norm") != "")
        .unique()
    )


def build_fish(dirname: str) -> pl.DataFrame:
    SKOS = "{http://www.w3.org/2004/02/skos/core#}"
    RDF = "{http://www.w3.org/1999/02/22-rdf-syntax-ns#}"
    XML_LANG = "{http://www.w3.org/XML/1998/namespace}lang"
    rows = []
    for concept in ET.parse(next((VOCAB_PATH / dirname).glob("*.rdf"))).getroot().iter(SKOS + "Concept"):
        subject = concept.get(RDF + "about").rstrip("/").rsplit("/", 1)[-1]
        for kind in ("prefLabel", "altLabel"):
            for el in concept.findall(SKOS + kind):
                rows.append((subject, el.text.strip(), el.get(XML_LANG), kind))
    return (
        pl.DataFrame(rows, schema=["subject", "term", "lang", "kind"], orient="row")
        .with_columns(norm=norm_term(pl.col("term")))
        .filter(pl.col("norm") != "")
        .unique()
    )


def build_periodo() -> pl.DataFrame:
    data = json.loads((VOCAB_PATH / "periodo/periodo-dataset.json").read_text())
    rows = []
    for authority in data["authorities"].values():
        for pid, period in authority.get("periods", {}).items():
            label = period.get("label")
            if label:
                rows.append((pid, label, (period.get("languageTag") or "").split("-")[0] or None,
                             "prefLabel"))
            for tag, alts in (period.get("localizedLabels") or {}).items():
                for alt in alts:
                    if alt != label:
                        rows.append((pid, alt, tag.split("-")[0], "altLabel"))
    return (
        pl.DataFrame(rows, schema=["subject", "term", "lang", "kind"], orient="row")
        .with_columns(norm=norm_term(pl.col("term")))
        .filter(pl.col("norm") != "")
        .unique()
    )


def build_tgn_children() -> pl.LazyFrame:
    """Direct narrower-place counts per TGN subject, the prominence prior"""
    return (
        pl.scan_csv(VOCAB_PATH / "tgn/TGNOut_HierarchicalRels.nt", separator="\x00",
                    has_header=False, new_columns=["line"], quote_char=None)
        .select(rel=pl.col("line").str.extract(r"tgn/rel/(\d+-broader-\d+)"))
        .drop_nulls()
        .unique()
        .select(subject=pl.col("rel").str.extract(r"-broader-(\d+)$"))
        .group_by("subject")
        .agg(prominence=pl.len().cast(pl.UInt32))
    )


def build_periodo_spatial() -> pl.DataFrame:
    """Spatial preference per period: 0 UK-covering, 1 Europe-covering, 2 other"""
    UK = {"united kingdom", "great britain", "britain", "england", "wales", "scotland",
          "northern ireland", "ireland", "british isles", "isle of man", "channel islands"}
    data = json.loads((VOCAB_PATH / "periodo/periodo-dataset.json").read_text())
    rows = []
    for authority in data["authorities"].values():
        periods = authority.get("periods", {})
        for pid, period in periods.items():
            labels = {c.get("label", "").lower()
                      for c in (period.get("spatialCoverage") or []) if isinstance(c, dict)}
            desc = (period.get("spatialCoverageDescription") or "").lower()
            uk = bool(labels & UK) or any(t in desc for t in UK)
            eu = "europe" in " ".join(labels) or "europe" in desc
            rows.append((pid, 0 if uk else (1 if eu else 2), -len(periods)))
    return pl.DataFrame(rows, schema={"subject": pl.String, "preference": pl.Int8,
                                      "pref_tiebreak": pl.Int32}, orient="row")


def cached(name: str, build) -> pl.LazyFrame:
    """Build once, cache as parquet, and hand back a lazy scan"""
    path = INDEX_DIR / f"{name}.parquet"
    if not path.exists():
        result = build()
        if isinstance(result, pl.LazyFrame):
            result.sink_parquet(path)
        else:
            result.write_parquet(path)
    return pl.scan_parquet(path)

In [4]:
with EmissionsTracker(project_name="vocab_index_build", output_dir=str(EMISSIONS_LOG_PATH), log_level="error") as tracker:
    INDEXES = {
        "aat": cached("aat", lambda: getty_term_index(VOCAB_PATH / "aat/AATOut_2Terms.nt")),
        "tgn": cached("tgn", lambda: getty_term_index(VOCAB_PATH / "tgn/TGNOut_2Terms.nt")),
        "ulan": cached("ulan", lambda: getty_term_index(VOCAB_PATH / "ulan/ULANOut_2Terms.nt")),
        "fish_building_materials": cached(
            "fish_building_materials", lambda: build_fish("fish_building_materials")),
        "fish_event_types": cached("fish_event_types", lambda: build_fish("fish_event_types")),
        "periodo": cached("periodo", build_periodo),
    }
    # Curated seeds and small vocabularies register through field_vocab_map
    INDEXES |= {
        vocab: cached(vocab, partial(load_termlist_index, vocab.removeprefix("local_")))
        for vocab in LOCAL_VOCABS
    }
    INDEXES |= {
        vocab: cached(vocab, STANDARD_BUILDERS[vocab]) for vocab in STANDARD_VOCABS
    }
    tgn_children = cached("tgn_children", build_tgn_children)
    periodo_spatial = cached("periodo_spatial", build_periodo_spatial)


def group_index(group: str) -> pl.LazyFrame:
    return pl.concat([
        INDEXES[vocab].with_columns(vocab=pl.lit(vocab), vocab_priority=pl.lit(priority))
        for priority, vocab in enumerate(GROUP_VOCABS[group])
    ])


for name, lf in INDEXES.items():
    print(f"{name}: {lf.select(pl.len()).collect(engine="streaming").item()}")

[codecarbon WARNING @ 23:30:15] Multiple instances of codecarbon are allowed to run at the same time.


aat: 604695
tgn: 8330116
ulan: 1616988
fish_building_materials: 672
fish_event_types: 172
periodo: 20920
local_persons_association: 257
local_field_collection_method: 49


### ULAN artefacts for `5_entities`

Agent type and preferred-biography years, joined on `subject` for tier-3 verification.

In [5]:
def build_ulan_agent_types() -> pl.LazyFrame:
    return (
        pl.scan_csv(VOCAB_PATH / "ulan/ULANOut_AgentTypes.nt", separator="\x00",
                    has_header=False, new_columns=["line"], quote_char=None)
        .select(rel=pl.col("line").str.extract(r"ulan/rel/(\d+-agentType-\d+)"))
        .drop_nulls()
        .unique()
        .select(
            subject=pl.col("rel").str.extract(r"^(\d+)"),
            agent_type_aat=pl.col("rel").str.extract(r"(\d+)$"),
        )
    )


def build_ulan_biographies() -> pl.DataFrame:
    lf = scan_nt(VOCAB_PATH / "ulan/ULANOut_Biographies.nt")
    preferred = (
        lf.filter(pl.col("p") == GVP + "biographyPreferred")
        .select(subject=pl.col("s").str.extract(r"/(\d+)$"),
                bio=pl.col("o").str.extract(r"bio/(\d+)"))
    )
    years = (
        lf.filter(pl.col("p").is_in([GVP + "estStart", GVP + "estEnd"]))
        .select(bio=pl.col("s").str.extract(r"bio/(\d+)"),
                pred=pl.col("p").str.extract(r"#(\w+)$"),
                year=pl.col("o").str.extract(r'^"(-?\d+)"').cast(pl.Int32))
        .collect(engine="streaming")
        .pivot("pred", index="bio", values="year", aggregate_function="first")
        .lazy()
    )
    return (
        preferred.join(years, on="bio")
        .select("subject", birth_year="estStart", death_year="estEnd")
        .collect(engine="streaming")
    )


ulan_agent_types = cached("ulan_agent_types", build_ulan_agent_types)
ulan_biographies = cached("ulan_biographies", build_ulan_biographies)
print(ulan_agent_types.select(pl.len()).collect(engine="streaming").item(), "agent types /",
      ulan_biographies.select(pl.len()).collect(engine="streaming").item(),
      "preferred biographies")

778922 agent types / 354003 preferred biographies


## 2. Distinct values per field group and institution

Fields sharing a target list share one cascade run. `data_source` is in the dedup key because separators are learned per institution.

In [6]:
CASCADE_FIELDS = list(GROUP_FOR)

value_ldf = (
    pl.scan_parquet(FIELD_STATS)
    .filter(pl.col("field_type").is_in(CASCADE_FIELDS) & pl.col("value").is_not_null())
    .select("record_id", "node_id", "data_source", "field_type", "value")
    .with_columns(pl.col("value").str.strip_chars().str.replace_all(r"\s+", " "))
    .filter(pl.col("value") != "")
    .with_columns(group=pl.col("field_type").replace_strict(GROUP_FOR, return_dtype=pl.String))
)

distinct = (
    value_ldf.group_by("group", "data_source", "value")
    # Splittable only where every occurrence sits in an atomiser-eligible field
    .agg(count=pl.len(),
         split_ok=(~pl.col("field_type").is_in(sorted(NO_ATOMISE_FIELDS))).all(),
         compound_ok=pl.col("field_type").is_in(sorted(COMPOUND_FIELDS)).all())
    .collect(engine="streaming")
)
distinct.group_by("group").agg(
    distinct_pairs=pl.len(), occurrences=pl.col("count").sum()
).sort("occurrences", descending=True)

group,distinct_pairs,occurrences
str,u32,u32
"""aat""",254430,11362572
"""tgn""",282560,7264887
"""aat+fish_building_materials""",98592,4102981
"""local_persons_association""",13042,2208608
"""periodo""",3258,343940
"""fish_event_types+local_field_c…",521,341895


## 3. Routing guards — null markers and prose

Null markers (`-`, `unknown`, `Place`) are `rejected` before they can exact-match real concepts. Prose, detected by length, passes whole to the deferral queue.

In [7]:
whole = distinct.with_columns(norm=norm_term(pl.col("value"))).with_columns(
    # Knowledge-state markers are kept as recorded but never cascaded
    route=pl.when(pl.col("norm").is_in(sorted(PLACEHOLDER_MARKERS)) | (pl.col("norm") == ""))
    .then(pl.lit("null_marker"))
    .when(pl.col("norm").is_in(sorted(SEMANTIC_MARKERS)))
    .then(pl.lit("semantic_marker"))
    # Amgueddfa Cymru's single-pipe values are bilingual pairs, not lists
    .when((pl.col("data_source") == "Amgueddfa Cymru - Museum Wales")
          & (pl.col("value").str.count_matches(r"\|") == 1)
          & pl.col("value").str.contains(r"\S\s*\|\s*\S"))
    .then(pl.lit("bilingual"))
    .when(pl.col("group").is_in(sorted(CONCEPT_GROUPS)) &
          ((pl.col("value").str.len_chars() > PROSE_MAX_CHARS) |
           (pl.col("value").str.split(" ").list.len() > PROSE_MAX_TOKENS)))
    .then(pl.lit("prose"))
    .otherwise(pl.lit("cascade"))
)
whole.group_by("group", "route").agg(
    values=pl.len(), occurrences=pl.col("count").sum()
).sort("group", "route")

group,route,values,occurrences
str,str,u32,u32
"""aat""","""bilingual""",1058,214245
"""aat""","""cascade""",250854,11138462
"""aat""","""null_marker""",69,3535
"""aat""","""prose""",2430,4570
"""aat""","""semantic_marker""",19,1760
…,…,…,…
"""periodo""","""semantic_marker""",4,368
"""tgn""","""bilingual""",12,2579
"""tgn""","""cascade""",282415,6821907


## 4. Delimiter induction per institution

A separator is real for an institution if splitting on it yields fragments attested as standalone values (≥ 5 occurrences) or as vocabulary terms, and it is promiscuous enough (`MIN_DISTINCT_FRAGMENTS`). `;` and `|` are baseline; `,`, `/`, `&`, `+`, ` and `, ` or ` are induced. Numeric guards keep `1,200` and `1/2` whole. Place values never atomise.

In [8]:
MIN_ATTEST_COUNT = 5        # occurrences for a whole value to count as attested
ATTEST_THRESHOLD = 0.5      # record-weighted share of fragments that must be attested
MIN_DISTINCT_FRAGMENTS = 10
MIN_SUPPORT_RECORDS = 200   # institution records containing the separator

concept_vals = whole.filter(
    (pl.col("route") == "cascade") & pl.col("group").is_in(sorted(CONCEPT_GROUPS)))

attested_corpus = (
    concept_vals.group_by("group", "norm").agg(pl.col("count").sum())
    .filter(pl.col("count") >= MIN_ATTEST_COUNT)
    .select("group", "norm")
)

# One frame of candidate splits: (group, data_source, count, separator, frags).
split_frames = []
for name, pattern in CANDIDATE_SEPARATORS.items():
    rx = re.compile(pattern)
    subset = concept_vals.filter(
        pl.col("value").str.contains(SEP_LITERAL[name], literal=True))
    if not len(subset):
        continue
    subset = subset.with_columns(
        separator=pl.lit(name),
        frags=pl.col("value").map_elements(
            lambda v, rx=rx: [p.strip() for p in rx.split(v) if p.strip()],
            return_dtype=pl.List(pl.String)),
    ).filter(pl.col("frags").list.len() >= 2)
    split_frames.append(subset.select("group", "data_source", "count", "separator", "frags"))

candidate_splits = pl.concat(split_frames)
exploded = (
    candidate_splits.explode("frags", empty_as_null=True)
    .rename({"frags": "frag"})
    .with_columns(frag_norm=norm_term(pl.col("frag")))
)

# A fragment attests as a frequent value or vocabulary term
frag_norms = exploded.select("group", "frag_norm").unique()
vocab_attested = pl.concat([
    frag_norms.filter(pl.col("group") == g).lazy()
    .join(group_index(g).select("norm").unique(), left_on="frag_norm", right_on="norm",
          how="semi")
    .collect(engine="streaming")
    for g in sorted(CONCEPT_GROUPS)
])
exploded = (
    exploded
    .join(attested_corpus.rename({"norm": "frag_norm"}).with_columns(c=pl.lit(True)),
          on=["group", "frag_norm"], how="left")
    .join(vocab_attested.with_columns(v=pl.lit(True)), on=["group", "frag_norm"], how="left")
    .with_columns(attested=pl.col("c").fill_null(False) | pl.col("v").fill_null(False))
)

sep_scores = (
    candidate_splits.group_by("group", "data_source", "separator")
    .agg(support_records=pl.col("count").sum(),
         frags_total=(pl.col("frags").list.len() * pl.col("count")).sum())
    .join(
        exploded
        .group_by("group", "data_source", "separator")
        .agg(
            frags_attested=(pl.col("attested").cast(pl.UInt32) * pl.col("count")).sum(),
            n_fragments=pl.col("frag_norm").filter(pl.col("attested")).n_unique(),
        ),
        on=["group", "data_source", "separator"],
    )
    .with_columns(attestation=pl.col("frags_attested") / pl.col("frags_total"))
    .with_columns(is_separator=(pl.col("attestation") >= ATTEST_THRESHOLD)
                  & (pl.col("n_fragments") >= MIN_DISTINCT_FRAGMENTS)
                  & (pl.col("support_records") >= MIN_SUPPORT_RECORDS))
    .sort("support_records", descending=True)
)
accepted = sep_scores.filter(pl.col("is_separator")).select("group", "data_source", "separator")

print(f"{len(accepted)} (group, institution, separator) acceptances out of {len(sep_scores)} scored candidates")
accepted.group_by("group", "separator").agg(institutions=pl.len()).sort("group", "separator")

127 (group, institution, separator) acceptances out of 660 scored candidates


group,separator,institutions
str,str,u32
"""aat""","""ampersand""",11
"""aat""","""and""",30
"""aat""","""comma""",14
"""aat""","""or""",2
"""aat""","""slash""",11
…,…,…
"""local_persons_association""","""and""",4
"""local_persons_association""","""comma""",1
"""local_persons_association""","""slash""",2


## 5. Atomisation fast path

Values whose whole norm exact-matches the vocabulary never split. Everything else splits on baseline plus induced separators with spans preserved. Long atoms defer as prose; multi-word atoms that fail every tier become the LLM queue.

In [9]:
cascade_vals = whole.filter(pl.col("route") == "cascade")

# Bilingual values cascade the English half over the whole span
bilingual = whole.filter(pl.col("route") == "bilingual").with_columns(
    en_atom=pl.col("value").str.split("|").list.last().str.strip_chars(),
    norm=norm_term(pl.col("value").str.split("|").list.last().str.strip_chars()),
)

# Whole-value exact hits, computed per group against the lazy index
whole_hits = pl.concat([
    cascade_vals.filter(pl.col("group") == g).lazy()
    .join(group_index(g).select("norm").unique(), on="norm", how="semi")
    .collect(engine="streaming")
    for g in GROUP_VOCABS
])

sep_sets = accepted.group_by("group", "data_source").agg(seps=pl.col("separator").sort())
to_split = (
    cascade_vals.filter(pl.col("group").is_in(sorted(CONCEPT_GROUPS))
                        & pl.col("split_ok")
                        & pl.col("value").str.contains(PREFILTER))
    .join(whole_hits.select("group", "data_source", "value"),
          on=["group", "data_source", "value"], how="anti")
    .join(sep_sets, on=["group", "data_source"], how="left")
)
no_split = (whole.filter(pl.col("route") != "bilingual")
            .join(to_split.select("group", "data_source", "value"),
                  on=["group", "data_source", "value"], how="anti"))

atoms = pl.concat([
    no_split.select(
        "group", "data_source", "value", "count", "route", "split_ok", "compound_ok",
        atom=pl.col("value"),
        span_start=pl.lit(0, dtype=pl.Int64),
        span_end=pl.col("value").str.len_chars().cast(pl.Int64),
    ),
    # Bilingual: English half is the atom, span covers everything
    bilingual.select(
        "group", "data_source", "value", "count",
        route=pl.lit("cascade"), split_ok=pl.col("split_ok"),
        compound_ok=pl.col("compound_ok"),
        atom=pl.col("en_atom"),
        span_start=pl.lit(0, dtype=pl.Int64),
        span_end=pl.col("value").str.len_chars().cast(pl.Int64),
    ),
    to_split.select(
        "group", "data_source", "value", "count", "route", "split_ok", "compound_ok",
        parts=pl.struct(["value", "seps"]).map_elements(
            lambda row: atomise(row["value"], row["seps"]),
            return_dtype=pl.List(pl.Struct({"atom": pl.String, "span_start": pl.Int64,
                                            "span_end": pl.Int64}))),
    ).explode("parts", empty_as_null=True).unnest("parts"),
]).with_columns(
    # A separator-only value explodes to null, so reject it
    empty_split=pl.col("atom").is_null(),
    atom=pl.col("atom").fill_null(pl.col("value")),
    span_start=pl.col("span_start").fill_null(0),
    span_end=pl.col("span_end").fill_null(pl.col("value").str.len_chars().cast(pl.Int64)),
).with_columns(norm=norm_term(pl.col("atom")))

# Null markers hide inside lists, and long atoms are sentences
atoms = atoms.with_columns(
    atom_route=pl.when(pl.col("route") != "cascade").then(pl.col("route"))
    .when(pl.col("empty_split") | (pl.col("norm") == "")
          | pl.col("norm").is_in(sorted(NULL_MARKERS)))
    .then(pl.lit("null_marker"))
    .when(pl.col("group").is_in(sorted(CONCEPT_GROUPS)) &
          ((pl.col("atom").str.len_chars() > ATOM_PROSE_MAX_CHARS) |
           (pl.col("atom").str.split(" ").list.len() > ATOM_PROSE_MAX_TOKENS)))
    .then(pl.lit("prose"))
    .otherwise(pl.lit("cascade"))
).drop("route", "empty_split")

print(f"{len(distinct)} distinct (group, institution, value) rows -> {len(atoms)} atoms "
      f"({len(to_split)} values split)")
atoms.group_by("group", "atom_route").agg(
    n=pl.len(), occurrences=pl.col("count").sum()).sort("group", "atom_route")

652403 distinct (group, institution, value) rows -> 902425 atoms (141068 values split)


group,atom_route,n,occurrences
str,str,u32,u32
"""aat""","""cascade""",397180,11952516
"""aat""","""null_marker""",82,3613
"""aat""","""prose""",4562,9396
"""aat""","""semantic_marker""",19,1760
"""aat+fish_building_materials""","""cascade""",199503,4495097
…,…,…,…
"""periodo""","""prose""",34,35
"""periodo""","""semantic_marker""",4,368
"""tgn""","""cascade""",282427,6824486


## 6. Tier 2a — exact alignment with homograph disambiguation

Lookup per group is the index restricted to observed norms. Homographs are settled by a ladder rather than flagged: **unique**, **kind_tier** (GVP descriptor > preferred > alternate), **spatial** (PeriodO, UK then Europe), **prominent** (TGN, ≥ 10 narrower places and ≥ 3× the runner-up). Unsettled cases stay `flagged` with the best candidate; `resolved_by` and `n_candidates` are kept for per-rung precision.

In [10]:
PROMINENCE_MIN = 10
PROMINENCE_RATIO = 3
DISAMBIG_SORT = {
    "by": ["preference", "pref_tiebreak", "prominence", "lang_p", "subject"],
    "descending": [False, False, True, False, False]
}


def resolve_norms(index: pl.LazyFrame, norms: pl.DataFrame | None,
                  prominence: pl.LazyFrame | None = None,
                  preference: pl.LazyFrame | None = None) -> pl.DataFrame:
    """Best (vocab, subject, term) per norm plus how any homograph was settled"""
    lf = index if norms is None else index.join(norms.lazy(), on="norm", how="semi")
    cand = (
        lf.with_columns(
            kind_p=pl.col("kind").replace_strict(KIND_PRIORITY, return_dtype=pl.Int8),
            lang_p=(~pl.col("lang").str.starts_with("en")).cast(pl.Int8).fill_null(1),
        )
        .group_by("norm", "vocab", "vocab_priority", "subject")
        .agg(
            kind_p=pl.col("kind_p").min(),
            lang_p=pl.col("lang_p").min(),
            matched_term=pl.col("term").sort_by(["kind_p", "lang_p"]).first(),
        )
    )
    cand = (cand.join(prominence, on="subject", how="left") if prominence is not None
            else cand.with_columns(prominence=pl.lit(0, dtype=pl.UInt32)))
    cand = (cand.join(preference, on="subject", how="left") if preference is not None
            else cand.with_columns(preference=pl.lit(0, dtype=pl.Int8), pref_tiebreak=pl.lit(0, dtype=pl.Int32)))
    cand = cand.with_columns(
        pl.col("prominence").fill_null(0),
        pl.col("preference").fill_null(2),
        pl.col("pref_tiebreak").fill_null(0))
    counts = cand.group_by("norm", "vocab", "vocab_priority").agg(
        n_candidates=pl.col("subject").n_unique().cast(pl.UInt32),
        min_kind=pl.col("kind_p").min())

    return (
        cand.join(counts, on=["norm", "vocab", "vocab_priority"])
        .filter(pl.col("kind_p") == pl.col("min_kind"))
        .group_by("norm", "vocab", "vocab_priority")
        .agg(
            n_candidates=pl.col("n_candidates").first(),
            n_best=pl.col("subject").n_unique(),
            subject=pl.col("subject").sort_by(**DISAMBIG_SORT).first(),
            matched_term=pl.col("matched_term").sort_by(**DISAMBIG_SORT).first(),
            n_pref_best=(pl.col("preference") == pl.col("preference").min()).sum(),
            pref_min=pl.col("preference").min(),
            prom_top=pl.col("prominence").sort(descending=True).head(2),
        )
        .with_columns(
            prom0=pl.col("prom_top").list.get(0, null_on_oob=True).fill_null(0),
            prom1=pl.col("prom_top").list.get(1, null_on_oob=True).fill_null(0),
        )
        .with_columns(
            resolved_by=pl.when(pl.col("n_candidates") == 1).then(pl.lit("unique"))
            .when(pl.col("n_best") == 1).then(pl.lit("kind_tier"))
            .when(pl.lit(preference is not None)
                  & ((pl.col("n_pref_best") == 1) | (pl.col("pref_min") == 0)))
            .then(pl.lit("spatial"))
            .when(pl.lit(prominence is not None) & (pl.col("prom0") >= PROMINENCE_MIN)
                  & (pl.col("prom0") >= PROMINENCE_RATIO * (pl.col("prom1") + 1)))
            .then(pl.lit("prominent")),
        )
        .sort("vocab_priority")
        .unique("norm", keep="first", maintain_order=True)
        .select("norm", "vocab", "subject", "matched_term", "n_candidates", "resolved_by")
        .collect(engine="streaming")
    )


def disambiguators(group: str) -> dict:
    kw = {}
    if group in PLACE_GROUPS:
        kw["prominence"] = tgn_children
    if "periodo" in GROUP_VOCABS[group]:
        kw["preference"] = periodo_spatial
    return kw


cascade_atoms = atoms.filter(pl.col("atom_route") == "cascade")
exact_hits = pl.concat([
    resolve_norms(group_index(g),
                  cascade_atoms.filter(pl.col("group") == g).select("norm").unique(),
                  **disambiguators(g))
    .with_columns(group=pl.lit(g), sub_component=pl.lit("exact"))
    for g in GROUP_VOCABS
])
tier_frames = [exact_hits]


def hit_norms() -> pl.DataFrame:
    return pl.concat([f.select("group", "norm") for f in tier_frames]).unique()


def pending_atoms() -> pl.DataFrame:
    return cascade_atoms.join(hit_norms(), on=["group", "norm"], how="anti")


exact_hits.group_by("group", "resolved_by").agg(norms=pl.len()).sort("group", "resolved_by")

group,resolved_by,norms
str,str,u32
"""aat""",null,2274
"""aat""","""kind_tier""",1740
"""aat""","""unique""",15659
"""aat+fish_building_materials""",null,869
"""aat+fish_building_materials""","""kind_tier""",664
…,…,…
"""periodo""","""unique""",102
"""tgn""",null,7666
"""tgn""","""kind_tier""",1356


### Tier 2a′ — British→American respelling retry

One deterministic GB→US pass on unmatched norms; a variant counts only if it exact-matches.

In [11]:
# TODO replace with utils function
GB_EXCEPTIONS = {
    "hour", "hours", "flour", "sour", "tour", "tours", "four", "pour", "pours", "our",
    "velour", "contour", "contours", "gourd", "gourds", "genre", "genres", "macabre",
    "turquoise", "praise", "rise", "wise", "paradise", "called", "rolled", "filled",
    "milled", "drilled", "spelled", "killed", "walled", "pulled", "chilled", "skilled",
}
GB_WORD_MAP = {
    "grey": "gray", "greys": "grays", "aluminium": "aluminum", "sulphur": "sulfur",
    "sulphate": "sulfate", "sulphide": "sulfide", "jewellery": "jewelry", "mould": "mold",
    "moulds": "molds", "moulded": "molded", "moulding": "molding", "mouldings": "moldings",
    "plough": "plow", "ploughs": "plows", "pyjamas": "pajamas", "tyre": "tire",
    "tyres": "tires", "kerb": "curb", "kerbs": "curbs", "storey": "story",
    "storeys": "stories", "gramme": "gram", "programme": "program",
    "programmes": "programs", "manoeuvre": "maneuver", "woollen": "woolen",
    "chequered": "checkered", "draught": "draft", "draughts": "drafts",
}
GB_RULES = [
    (re.compile(r"our(s|ed|ing|ings)?$"), lambda m: "or" + (m.group(1) or "")),
    (re.compile(r"(?<=[tbh])re(s)?$"), lambda m: "er" + (m.group(1) or "")),
    (re.compile(r"(?<=[lnrmtvdg])is(e|es|ed|ing|ation|ations)$"), lambda m: "iz" + m.group(1)),
    (re.compile(r"(?<=[a-z])ae"), lambda m: "e"),
    (re.compile(r"ll(ed|ing|er|ers)$"), lambda m: "l" + m.group(1)),
]


def us_variant(norm: str) -> str | None:
    out, changed = [], False
    for tok in norm.split(" "):
        if tok in GB_WORD_MAP:
            out.append(GB_WORD_MAP[tok])
            changed = True
            continue
        if tok in GB_EXCEPTIONS:
            out.append(tok)
            continue
        v = tok
        for rx, rep in GB_RULES:
            v = rx.sub(rep, v)
        out.append(v)
        changed |= v != tok
    return " ".join(out) if changed else None


variant_frames = []
for g in sorted(CONCEPT_GROUPS):
    pending = pending_atoms().filter(pl.col("group") == g).select("norm").unique()
    pairs = pl.DataFrame({
        "norm": pending["norm"],
        "variant": [us_variant(n) if n else None for n in pending["norm"]],
    }).drop_nulls("variant")
    if not len(pairs):
        continue
    lookup = resolve_norms(group_index(g), pairs.select(norm=pl.col("variant")).unique())
    hits = (pairs.join(lookup.rename({"norm": "variant"}), on="variant")
            .drop("variant")
            .with_columns(group=pl.lit(g), sub_component=pl.lit("exact_variant")))
    variant_frames.append(hits)

variant_hits = (pl.concat(variant_frames) if variant_frames else exact_hits.clear())
tier_frames.append(variant_hits)
variant_hits.group_by("group").agg(norms=pl.len()).sort("norms", descending=True)

group,norms
str,u32
"""aat""",91
"""aat+fish_building_materials""",37


### Tier 2a″ — parenthetical-qualifier retry

`metal (unknown)`, `film (photographic)`: the head is matched alone and the qualifier kept in the sidecar.

In [12]:
PAREN = r"^(.+?) ?\((.+)\)$"

paren_frames = []
for g in sorted(CONCEPT_GROUPS):
    pending = (
        pending_atoms().filter(pl.col("group") == g).select("norm").unique()
        .with_columns(head=pl.col("norm").str.extract(PAREN, 1),
                      qualifier=pl.col("norm").str.extract(PAREN, 2))
        .drop_nulls("head")
        .with_columns(head=norm_term(pl.col("head")))
        .filter(pl.col("head") != "")
    )
    if not len(pending):
        continue
    lookup = resolve_norms(group_index(g), pending.select(norm=pl.col("head")).unique())
    hits = (pending.join(lookup.rename({"norm": "head"}), on="head")
            .drop("head")
            .with_columns(group=pl.lit(g), sub_component=pl.lit("exact_paren")))
    paren_frames.append(hits)

paren_hits = (pl.concat(paren_frames) if paren_frames else
              exact_hits.clear().with_columns(qualifier=pl.lit(None, dtype=pl.String)))
tier_frames.append(paren_hits)
paren_hits.group_by("group").agg(norms=pl.len()).sort("norms", descending=True)

group,norms
str,u32
"""aat""",2843
"""aat+fish_building_materials""",2540
"""local_persons_association""",164
"""periodo""",151
"""fish_event_types+local_field_c…",24


### Tier 2a‴ — deverbal-morphology retry

`engraved` → `engraving`, `dyed` → `dyeing`: one `-ed` → `-ing` pass, anchored on the vocabulary.

In [13]:
morph_frames = []
for g in sorted(CONCEPT_GROUPS):
    pending = pending_atoms().filter(pl.col("group") == g).select("norm").unique()
    pairs = (pl.DataFrame({"norm": pending["norm"]})
             .with_columns(variant=pl.col("norm").map_elements(
                 morph_variants, return_dtype=pl.List(pl.String)))
             .explode("variant").drop_nulls("variant")
             .with_row_index("prio"))
    if not len(pairs):
        continue
    lookup = resolve_norms(group_index(g), pairs.select(norm=pl.col("variant")).unique())
    hits = (pairs.join(lookup.rename({"norm": "variant"}), on="variant")
            .sort("prio").unique("norm", keep="first")
            .drop("variant", "prio")
            .with_columns(group=pl.lit(g), sub_component=pl.lit("exact_morph")))
    morph_frames.append(hits)

morph_hits = (pl.concat(morph_frames) if morph_frames else exact_hits.clear())
tier_frames.append(morph_hits)
morph_hits.group_by("group").agg(norms=pl.len()).sort("norms", descending=True)

/tmp/ipykernel_2347752/1970889387.py:7: DeprecationWarning: In Polars 2.0, the default behavior for `empty_as_null` will change to `False`. To keep the current behavior, explicitly set `empty_as_null=True`.
  .explode("variant").drop_nulls("variant")


group,norms
str,u32
"""aat""",168
"""aat+fish_building_materials""",37
"""fish_event_types+local_field_c…",2


### Tier 2a⁗ — compositional head retry

`engraving on paper`, `ink on paper`: technique plus support. The head is matched alone and the remainder kept as qualifier, only where both sides exact-match the group's index and every occurrence sits in a `COMPOUND_FIELDS` field. `material` is excluded because there `ink on paper` names two materials.

In [ ]:
compound_frames = []
for g in sorted(CONCEPT_GROUPS):
    pending = (
        pending_atoms().filter(pl.col("group") == g)
        .group_by("norm").agg(compound_ok=pl.col("compound_ok").all())
        .filter("compound_ok")
        .with_columns(parts=pl.col("norm").map_elements(
            compound_head, return_dtype=pl.List(pl.String)))
        .drop_nulls("parts")
        .select("norm",
                head=pl.col("parts").list.get(0),
                tail=pl.col("parts").list.get(1),
                qualifier=pl.col("parts").list.get(2))
    )
    if not len(pending):
        continue
    index = group_index(g)
    # the tail must attest too, proving a composition
    attested = index.select("norm").unique().collect(engine="streaming")["norm"]
    pending = pending.filter(pl.col("tail").is_in(attested.implode()))
    if not len(pending):
        continue
    lookup = resolve_norms(index, pending.select(norm=pl.col("head")).unique())
    hits = (pending.join(lookup.rename({"norm": "head"}), on="head")
            .drop("head", "tail")
            .with_columns(group=pl.lit(g), sub_component=pl.lit("exact_compound")))
    compound_frames.append(hits)

compound_hits = (pl.concat(compound_frames) if compound_frames else
                 exact_hits.clear().with_columns(qualifier=pl.lit(None, dtype=pl.String)))
tier_frames.append(compound_hits)
compound_hits.group_by("group").agg(norms=pl.len()).sort("norms", descending=True)

### Tier 2c — house vocabularies, read before the pooled targets

Institutions `institutional_vocab_detect.py` credits with a published list (BM Materials, SHIC) have that list read first through the same ladder. Hits key on `(group, data_source, norm)`. A verbatim pooled match survives as the crosswalk (§10). `date_period` stays unwired because HE Periods is a label subset of PeriodO.

In [ ]:
from mds_norm.pipeline.vocab_alignment import house_ranks, house_tier

print(house_ranks().group_by("group", "vocab").agg(institutions=pl.len()).sort("group", "vocab"))

house_hits = house_tier(cascade_atoms)
(house_hits.group_by("group", "vocab", "sub_component")
 .agg(institutions=pl.col("data_source").n_unique(), norms=pl.len())
 .sort("group", "vocab", "sub_component"))

## 7. Tier 2b — fuzzy alignment

Indel ratio against the full English vocabulary of each group, chunked, with a high cutoff and minimum length. Places skip this tier.

In [15]:
FUZZY_ACCEPT = 93.0
FUZZY_MIN_LEN = 4
CHUNK = 2_000


def english_only(index: pl.LazyFrame) -> pl.LazyFrame:
    return index.filter(pl.col("lang").str.starts_with("en").fill_null(True))


fuzzy_frames = []
with EmissionsTracker(project_name="vocab_align_fuzzy", output_dir=str(EMISSIONS_LOG_PATH), log_level="error") as tracker:
    for group in sorted(CONCEPT_GROUPS):
        pending = (
            pending_atoms()
            .filter((pl.col("group") == group)
                    & (pl.col("norm").str.len_chars() >= FUZZY_MIN_LEN))
            .select("norm").unique()
        )
        table = resolve_norms(english_only(group_index(group)), None)
        choices = table["norm"].to_list()
        queries = pending["norm"].to_list()
        if not queries:
            continue
        best_idx, best_score = [], []
        for i in range(0, len(queries), CHUNK):
            scores = cdist(queries[i:i + CHUNK], choices, scorer=fuzz.ratio, score_cutoff=FUZZY_ACCEPT, workers=-1, dtype=np.uint8)
            best_idx.append(scores.argmax(axis=1))
            best_score.append(scores.max(axis=1))
        best_idx, best_score = np.concatenate(best_idx), np.concatenate(best_score)
        hits = (
            pl.DataFrame({"norm": queries, "match_norm": [choices[j] for j in best_idx],
                          "score": best_score.astype(np.float64)})
            .filter(pl.col("score") >= FUZZY_ACCEPT)
            .join(table.rename({"norm": "match_norm"}), on="match_norm")
            .with_columns(group=pl.lit(group), sub_component=pl.lit("fuzzy"),
                          score=pl.col("score") / 100.0)
            .drop("match_norm")
        )
        fuzzy_frames.append(hits)

fuzzy_hits = (pl.concat(fuzzy_frames) if fuzzy_frames else
              exact_hits.clear().with_columns(score=pl.lit(None, dtype=pl.Float64)))
tier_frames.append(fuzzy_hits)
fuzzy_hits.group_by("group").agg(matched=pl.len()).sort("matched", descending=True)

group,matched
str,u32
"""aat""",3257
"""aat+fish_building_materials""",1063
"""local_persons_association""",173
"""periodo""",70
"""fish_event_types+local_field_c…",12


## 8. Tier 4 — zero-shot LLM atomiser

A small instruct model proposes verbatim substrings for values the deterministic splitter could not decompose. Atoms must be contiguous substrings of at most 5 words; survivors re-enter the exact tier. Queue capped at 95% of each group's deferred occurrence mass. Date expressions in `date_period` route to the dates pipeline instead.

In [17]:
for idx_name, idx in INDEXES.items():
    quant_95 = idx.select(pl.col("term").str.split(" ").list.len().quantile(0.95)).collect(engine="streaming").item()
    print(f'{idx_name}: {quant_95}', end=" | ")

aat: 4.0 | tgn: 3.0 | ulan: 4.0 | fish_building_materials: 3.0 | fish_event_types: 5.0 | periodo: 5.0 | local_persons_association: 2.0 | local_field_collection_method: 2.0 | 

In [18]:
queue = (
    pending_atoms()
    .filter(pl.col("group").is_in(sorted(CONCEPT_GROUPS))
            & pl.col("split_ok")  # no-atomise fields never reach the LLM splitter
            & pl.col("atom").str.contains(" ", literal=True)
            & ~((pl.col("group") == "periodo") & pl.col("norm").str.contains(DATE_LIKE)))
    .group_by("group", "atom")
    .agg(occ=pl.col("count").sum())
    .sort("occ", descending=True)
    .with_columns(cum_share=(pl.col("occ").cum_sum() / pl.col("occ").sum()).over("group"))
)
llm_queue = queue.filter(pl.col("cum_share") <= LLM_COVERAGE)
print(f"LLM queue: {len(llm_queue):,} of {len(queue):,} distinct atoms "
      f"({llm_queue['occ'].sum():,} of {queue['occ'].sum():,} occurrences)")

LLM queue: 68,878 of 129,106 distinct atoms (1,154,769 of 1,215,550 occurrences)


In [19]:
LLM_MODEL = "LFM2.5-350M"
LLM_API_BASE = "http://localhost:30000/v1"
LLM_CONCURRENCY = 256
# https://docs.vllm.ai/projects/recipes/en/latest/LiquidAI/LFM2.5.html#recommended-sampling
TEMPERATURE = 0.1
EXTRA_BODY = {"top_k": 50, "repetition_penalty": 1.05}

inf = Inference(model=LLM_MODEL, base_url=LLM_API_BASE, concurrency=LLM_CONCURRENCY, timeout=600.0)

atomise_samples = [
    {"desc": GROUP_DESC[g], "value": v}
    for g, v in zip(llm_queue["group"], llm_queue["atom"])
]

with EmissionsTracker(project_name="vocab_atomise_llm", output_dir=str(EMISSIONS_LOG_PATH),
                      log_level="error", tracking_mode="machine"):
    completions = await inf.generate(
        atomise_samples,
        LLM_PROMPT,
        max_tokens=LLM_MAX_NEW_TOKENS,
        temperature=TEMPERATURE,
        extra_body=EXTRA_BODY
    )

llm_rows = []
for g, v, c in zip(llm_queue["group"], llm_queue["atom"], completions):
    for atom in parse_llm_atoms(v, c or "") or []:
        llm_rows.append({"group": g, "atom": v, **atom})

llm_split = (pl.DataFrame(llm_rows) if llm_rows else
             pl.DataFrame(schema={"group": pl.String, "atom": pl.String,
                                  "sub_atom": pl.String, "sub_start": pl.Int64,
                                  "sub_end": pl.Int64}))
llm_processed = llm_queue.select("group", "atom")
print(f"{llm_split['atom'].n_unique() if len(llm_split) else 0:,} of {len(llm_queue):,} "
      f"queued atoms split into {len(llm_split):,} sub-atoms")

Output()

31,041 of 68,878 queued atoms split into 86,304 sub-atoms


In [20]:
# LLM sub-atoms re-enter the exact tier only, nothing fuzzier
llm_split = llm_split.with_columns(sub_norm=norm_term(pl.col("sub_atom")))

llm_hit_frames = []
for g in sorted(CONCEPT_GROUPS):
    norms = (llm_split.filter(pl.col("group") == g).select(norm=pl.col("sub_norm")).unique())
    if not len(norms):
        continue
    lookup = resolve_norms(group_index(g), norms).with_columns(
        group=pl.lit(g), sub_component=pl.lit("llm_exact"))
    llm_hit_frames.append(lookup)
    unmatched = norms.join(lookup.select("norm"), on="norm", how="anti")
    pairs = pl.DataFrame({
        "norm": unmatched["norm"],
        "variant": [us_variant(n) if n else None for n in unmatched["norm"]],
    }).drop_nulls("variant")
    if len(pairs):
        vlookup = resolve_norms(group_index(g), pairs.select(norm=pl.col("variant")).unique())
        llm_hit_frames.append(
            pairs.join(vlookup.rename({"norm": "variant"}), on="variant").drop("variant")
            .with_columns(group=pl.lit(g), sub_component=pl.lit("llm_variant")))
    mp = (pl.DataFrame({"norm": unmatched["norm"]})
          .with_columns(variant=pl.col("norm").map_elements(
              morph_variants, return_dtype=pl.List(pl.String)))
          .explode("variant", empty_as_null=True).drop_nulls("variant").with_row_index("prio"))
    if len(mp):
        mlookup = resolve_norms(group_index(g), mp.select(norm=pl.col("variant")).unique())
        llm_hit_frames.append(
            mp.join(mlookup.rename({"norm": "variant"}), on="variant")
            .sort("prio").unique("norm", keep="first").drop("variant", "prio")
            .with_columns(group=pl.lit(g), sub_component=pl.lit("llm_morph")))

llm_lookup = (pl.concat(llm_hit_frames).unique(["group", "norm"], keep="first")
              if llm_hit_frames else exact_hits.clear())
llm_lookup.group_by("group", "sub_component").agg(norms=pl.len()).sort("group")

group,sub_component,norms
str,str,u32
"""aat""","""llm_morph""",71
"""aat""","""llm_exact""",4185
"""aat""","""llm_variant""",24
"""aat+fish_building_materials""","""llm_variant""",12
"""aat+fish_building_materials""","""llm_exact""",2102
"""aat+fish_building_materials""","""llm_morph""",21
"""fish_event_types+local_field_c…","""llm_exact""",9
"""local_persons_association""","""llm_exact""",90
"""periodo""","""llm_exact""",48


## 9. Tier 4b — candidate retrieval and a model that selects one

Replaces the cosine-only rung (0.467 precision on homograph gold). `LFM2.5-Embedding-350M` retrieves the top five subjects; a model shown the value and the candidates with one line of context each answers with an option number or 0 for none. Runs after the atomiser and writes `flagged` until scored against gold.

In [ ]:
from mds_norm.pipeline.vocab_alignment import rerank_pending
from mds_norm.pipeline.vocab_rerank import rerank_tier

rerank_hits, rerank_queued = rerank_tier(rerank_pending(pending_atoms(), llm_split, llm_lookup))
tier_frames.append(rerank_hits)

# hits reach split sub-atoms and whole atoms alike
llm_lookup = pl.concat([llm_lookup, rerank_hits], how="diagonal").unique(
    ["group", "norm"], keep="first", maintain_order=True)

print(f"{len(rerank_hits):,} of {len(rerank_queued):,} queued atoms selected a candidate")
rerank_hits.group_by("group").agg(
    selected=pl.len(), median_cosine=pl.col("score").median()
).sort("selected", descending=True) if len(rerank_hits) else rerank_hits

## 10. Decisions and sidecar assembly

Tier results fold onto the atom table in priority order (exact > variant > paren > fuzzy > rerank); LLM-split atoms are replaced by their sub-atoms. Status: `resolved`, `flagged`, `rejected`, or `deferred` with a routing reason. House hits overlay per `(group, data_source, norm)` before LLM-parent replacement; a displaced verbatim pooled match stays as `xref_vocab` / `xref_subject`. Decisions join to every occurrence lazily and sink to parquet.

In [21]:
tier_hits = pl.concat(
    [f for f in tier_frames if len(f)], how="diagonal"
).unique(["group", "norm"], keep="first", maintain_order=True)

base = atoms.join(tier_hits, on=["group", "norm"], how="left")

# House list displaces the pooled match, applied before splitting
from mds_norm.pipeline.vocab_alignment import house_overlay

base = house_overlay(base, house_hits)

# LLM-split parents never hit, so use their sub-atoms
kept = base.join(llm_split.select("group", "atom").unique().with_columns(rep=pl.lit(True)),
                 on=["group", "atom"], how="left")
llm_parent_rows = kept.filter(pl.col("rep").is_not_null() & pl.col("subject").is_null()
                              & pl.col("split_ok"))
kept = kept.filter(pl.col("rep").is_null() | pl.col("subject").is_not_null()
                   | ~pl.col("split_ok")).drop("rep")

llm_atom_rows = (
    llm_parent_rows
    .select("group", "data_source", "value", "count", "atom_route", "atom",
            "split_ok", parent_start=pl.col("span_start"))
    .join(llm_split.drop("sub_norm"), on=["group", "atom"])
    .with_columns(
        atom=pl.col("sub_atom"),
        span_start=pl.col("parent_start") + pl.col("sub_start"),
        span_end=pl.col("parent_start") + pl.col("sub_end"),
        norm=norm_term(pl.col("sub_atom")),
    )
    .drop("sub_atom", "sub_start", "sub_end", "parent_start")
    .join(llm_lookup, on=["group", "norm"], how="left")
    .with_columns(from_llm=pl.lit(True))
)

decisions = (
    pl.concat([kept, llm_atom_rows], how="diagonal")
    .join(llm_processed.with_columns(llm_done=pl.lit(True)),
          on=["group", "atom"], how="left")
    .with_columns(
        status=pl.when(pl.col("atom_route") == "null_marker").then(pl.lit("rejected"))
        .when(pl.col("subject").is_null()).then(pl.lit("deferred"))
        .when(pl.col("resolved_by").is_null()).then(pl.lit("flagged"))
        .otherwise(pl.lit("resolved")),
    )
    .with_columns(
        defer_reason=pl.when(pl.col("status") != "deferred")
        .then(pl.lit(None, dtype=pl.String))
        .when(pl.col("atom_route") == "prose").then(pl.lit("prose"))
        .when(pl.col("atom_route") == "semantic_marker")
        .then(pl.lit("semantic_marker"))
        .when(pl.col("group").is_in(sorted(PLACE_GROUPS))).then(pl.lit("place_pipeline"))
        .when((pl.col("group") == "periodo") & pl.col("norm").str.contains(DATE_LIKE))
        .then(pl.lit("date_parser"))
        .when(pl.col("llm_done").fill_null(False) | pl.col("from_llm").fill_null(False))
        .then(pl.lit("no_match"))
        .when(pl.col("atom").str.contains(" ", literal=True)).then(pl.lit("atomiser"))
        .otherwise(pl.lit("no_match")),
        component=pl.lit(COMPONENT),
        tier=pl.when(pl.col("sub_component").str.starts_with("llm")).then(4)
        .when(pl.col("sub_component") == "rerank").then(4)
        .when(pl.col("sub_component").is_not_null()).then(2),
        confidence=pl.when(pl.col("status") == "resolved")
        .then(pl.col("sub_component").replace_strict(CONFIDENCE, return_dtype=pl.Float64,
                                                     default=None)
              * pl.col("resolved_by").replace_strict(RESOLVED_BY_FACTOR,
                                                     return_dtype=pl.Float64, default=1.0)),
    )
    .drop("llm_done", "from_llm")
)
decisions.write_parquet(OUT_DIR / "vocab_value_decisions.parquet")
print(f"{len(decisions):,} atom decisions -> {OUT_DIR / 'vocab_value_decisions.parquet'}")

decisions.group_by("group", "status").agg(
    atoms=pl.len(), occurrences=pl.col("count").sum()
).sort("group", "status")

983,014 atom decisions -> /home/liam/Documents/university/mds-norm/notebooks/analysis_output/vocabularies/vocab_value_decisions.parquet


group,status,atoms,occurrences
str,str,u32,u32
"""aat""","""deferred""",203512,3210606
"""aat""","""flagged""",71758,2136462
"""aat""","""rejected""",82,3613
"""aat""","""resolved""",186861,6917412
"""aat+fish_building_materials""","""deferred""",35603,195224
…,…,…,…
"""periodo""","""resolved""",912,219091
"""tgn""","""deferred""",232227,2355226
"""tgn""","""flagged""",22727,1853272


In [22]:
# Apply scored rung verdicts as a join, not a recompute
from mds_norm.pipeline.apply_homograph_verdicts import RUNG_VERDICTS, apply_verdicts

if RUNG_VERDICTS.exists():
    decisions = apply_verdicts(decisions)
    decisions.write_parquet(OUT_DIR / "vocab_value_decisions.parquet")
    print(decisions.group_by("status").agg(atoms=pl.len(),
                                           occurrences=pl.col("count").sum()).sort("status"))

rung verdicts applied: {'flagged_best': 'keep_flagged', 'fuzzy': 'demote', 'kind_tier': 'demote', 'prominent': 'keep', 'semantic': 'demote', 'spatial': 'keep'}
shape: (4, 3)
┌──────────┬────────┬─────────────┐
│ status   ┆ atoms  ┆ occurrences │
│ ---      ┆ ---    ┆ ---         │
│ str      ┆ u32    ┆ u32         │
╞══════════╪════════╪═════════════╡
│ deferred ┆ 485441 ┆ 6157435     │
│ flagged  ┆ 272036 ┆ 9396037     │
│ rejected ┆ 308    ┆ 460240      │
│ resolved ┆ 225229 ┆ 11100760    │
└──────────┴────────┴─────────────┘


In [23]:
(
    value_ldf
    .join(decisions.drop("count", "norm", "atom_route", "split_ok").lazy(),
          on=["group", "data_source", "value"], how="left")
    .sink_parquet(OUT_DIR / "vocab_annotations.parquet")
)
ann = pl.scan_parquet(OUT_DIR / "vocab_annotations.parquet")
print(f"{ann.select(pl.len()).collect(engine="streaming").item():,} annotation rows -> {OUT_DIR / 'vocab_annotations.parquet'}")

27,114,472 annotation rows -> /home/liam/Documents/university/mds-norm/notebooks/analysis_output/vocabularies/vocab_annotations.parquet


## 11. Coverage — occurrence- and type-weighted per institution

Linkage per field and per institution, occurrence- and distinct-weighted. `rejected` is excluded from the denominator; `flagged` counts separately.

In [24]:
per_field = (
    ann.filter(pl.col("status") != "rejected")
    .group_by("field_type")
    .agg(
        occurrences=pl.len(),
        resolved=(pl.col("status") == "resolved").mean(),
        flagged=(pl.col("status") == "flagged").mean(),
        deferred=(pl.col("status") == "deferred").mean(),
    )
    .sort("resolved")
    .collect(engine="streaming")
)
per_field.with_columns(meets_target=pl.col("resolved") >= 0.85)

field_type,occurrences,resolved,flagged,deferred,meets_target
str,u32,f64,f64,f64,bool
"""spectrum/object_production_pla…",2172955,0.292088,0.430507,0.277404,false
"""spectrum/technique""",3407582,0.31,0.494334,0.195666,false
"""spectrum/field_collection_plac…",2937494,0.321069,0.306261,0.37267,false
"""spectrum/material""",4643877,0.353734,0.604227,0.042039,false
"""spectrum/object_component_name""",601027,0.380124,0.556346,0.06353,false
…,…,…,…,…,…
"""spectrum/content_concept""",811145,0.451458,0.422514,0.126028,false
"""spectrum/inscription_type""",246163,0.472845,0.420526,0.106629,false
"""spectrum/date_period""",355170,0.581797,0.137255,0.280947,false


In [25]:
active = ann.filter(pl.col("status") != "rejected")
per_institution = (
    active.group_by("data_source", "group")
    .agg(occurrences=pl.len(), resolved_occ=(pl.col("status") == "resolved").mean())
    .collect(engine="streaming")
    .join(
        active.unique(["data_source", "group", "value"])
        .group_by("data_source", "group")
        .agg(distinct=pl.len(), resolved_distinct=(pl.col("status") == "resolved").mean())
        .collect(engine="streaming"),
        on=["data_source", "group"],
    )
    .sort("resolved_occ")
)

below_target = per_institution.filter(pl.col("resolved_occ") < 0.85)
print(f"{len(below_target)}/{len(per_institution)} (institution, group) pairs below the "
      f"85% occurrence-weighted target")
per_institution.head(20)

269/330 (institution, group) pairs below the 85% occurrence-weighted target


data_source,group,occurrences,resolved_occ,distinct,resolved_distinct
enum,str,u32,f64,u32,f64
"""Wotton-under-Edge Heritage Cen…","""tgn""",3,0.0,3,0.0
"""National Museums Liverpool""","""aat""",77346,0.0,93,0.0
"""Nothe Fort""","""aat""",637,0.0,1,0.0
"""Cyfarthfa Castle Museum & Art …","""fish_event_types+local_field_c…",2,0.0,2,0.0
"""Rushden Transport Museum""","""periodo""",11,0.0,4,0.0
…,…,…,…,…,…
"""National Gallery""","""periodo""",3301,0.0,32,0.0
"""Fry Art Gallery""","""tgn""",1,0.0,1,0.0
"""Ravenglass Railway Museum""","""periodo""",6960,0.000144,151,0.006623


In [26]:
residue_worklist = (
    decisions.filter(pl.col("status") == "deferred")
    .with_columns(
        shape=pl.col("atom").str.replace_all(r"[A-Za-z]+", "s").str.replace_all(r"\d+", "d"))
    .group_by("group", "defer_reason", "shape")
    .agg(distinct=pl.len(), occurrences=pl.col("count").sum(), examples=pl.col("atom").head(3))
    .sort("occurrences", descending=True)
)
residue_worklist.head(20)

group,defer_reason,shape,distinct,occurrences,examples
str,str,str,u32,u32,list[str]
"""aat""","""atomiser""","""s s""",15854,520584,"[""gas services"", ""River Skell"", ""Pedigree Toys""]"
"""tgn""","""place_pipeline""","""s s""",27876,474033,"[""Colchester Harwood"", ""Victoria Hotel"", ""Cliffe Villas""]"
"""aat""","""no_match""","""s""",24992,356931,"[""Opium"", ""fragilis"", ""agorigine""]"
"""aat""","""no_match""","""s s""",32765,318344,"[""back yard"", ""Sepia drawing"", ""Country Life""]"
"""aat""","""atomiser""","""s s s""",9262,248166,"[""Naval Intelligence Division"", ""flying boot warmer"", ""sepia watercolour painting""]"
…,…,…,…,…,…
"""tgn""","""place_pipeline""","""s""",10128,96769,"[""Docimeium"", ""Drumgooland"", ""Penznace""]"
"""aat""","""no_match""","""ds""",103,91813,"[""1960s"", ""1990s"", ""1810s""]"
"""aat""","""atomiser""","""s s s s.s. s""",5,79108,"[""record verified by S.A. Fox"", ""record verified by E.A. Walker"", ""record verified by R.J. Brewer""]"


## 12. Review sample

Head + tail per `(group, sub_component, resolved_by)` over resolved and flagged atoms, so each rung and retry gets its own precision estimate.

In [27]:
REVIEW_N = 12

reviewable = decisions.filter(pl.col("status").is_in(["resolved", "flagged"]))
review = pl.concat([
    pl.concat([
        part.sort("count", descending=True).head(REVIEW_N),
        part.sample(min(REVIEW_N, len(part)), seed=0),
    ])
    for _, part in reviewable.group_by("group", "sub_component", "resolved_by")
]).unique(["group", "atom"], maintain_order=True)

review_cols = ["group", "sub_component", "resolved_by", "status", "data_source", "value",
               "atom", "qualifier", "vocab", "subject", "matched_term", "score",
               "n_candidates", "count"]
review.select(review_cols).write_csv(OUT_DIR / "vocab_review_sample.csv")
print(len(review), "review rows ->", OUT_DIR / "vocab_review_sample.csv")
review.select(review_cols).head(20)

1118 review rows -> /home/liam/Documents/university/mds-norm/notebooks/analysis_output/vocabularies/vocab_review_sample.csv


group,sub_component,resolved_by,status,data_source,value,atom,qualifier,vocab,subject,matched_term,score,n_candidates,count
str,str,str,str,enum,str,str,str,str,str,str,f64,u32,u32
"""tgn""","""exact""","""prominent""","""resolved""","""Norfolk Museums Service""","""England""","""England""",null,"""tgn""","""7002445""","""England""",null,9,177502
"""tgn""","""exact""","""prominent""","""resolved""","""Wiltshire Museum""","""Wiltshire""","""Wiltshire""",null,"""tgn""","""7008180""","""Wiltshire""",null,4,122392
"""tgn""","""exact""","""prominent""","""resolved""","""Norfolk Museums Service""","""Norfolk""","""Norfolk""",null,"""tgn""","""7008160""","""Norfolk""",null,15,86620
"""tgn""","""exact""","""prominent""","""resolved""","""Victoria and Albert Museum""","""France""","""France""",null,"""tgn""","""1000070""","""France""",null,6,49586
"""tgn""","""exact""","""prominent""","""resolved""","""Amgueddfa Cymru - Museum Wales""","""Gwynedd""","""Gwynedd""",null,"""tgn""","""7029493""","""Gwynedd""",null,2,27062
…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""tgn""","""exact""","""prominent""","""resolved""","""Potteries Museum & Art Gallery""","""Bologna""","""Bologna""",null,"""tgn""","""7003127""","""Bologna""",null,2,1
"""tgn""","""exact""","""prominent""","""resolved""","""Norfolk Museums Service""","""Hong Kong""","""Hong Kong""",null,"""tgn""","""7004542""","""Hong Kong""",null,3,107
"""tgn""","""exact""","""prominent""","""resolved""","""Southampton Cultural Services""","""Hampshire""","""Hampshire""",null,"""tgn""","""7008139""","""Hampshire""",null,11,8174
